### Import

In [40]:
from base_utils import load_test, load_train, numeric_columns, categorical_columns, selected_features, categorical

from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, random_split

### Data Preprocess

In [2]:
train_df = load_train()
test_df = load_test()

In [3]:
train_df[numeric_columns] = train_df[numeric_columns].fillna(0)
test_df[numeric_columns] = test_df[numeric_columns].fillna(0)

In [4]:
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

In [5]:
# 범주형 컬럼들 문자열로 변환(데이터 타입 일관성 유지) + 결측치 처리
for col in categorical_columns:
    X[col] = X[col].astype(str).fillna('Unknown')
    test_df[col] = test_df[col].astype(str).fillna('Unknown')

In [6]:
# 서열이 없으므로 레이블 인코딩
# Label Encoding을 위한 딕셔너리 (각 컬럼별로 인코더 저장)
label_encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    # 훈련 데이터에 대해 Label Encoding 수행
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le  # 인코더 저장
    
    # 각 컬럼별 mapping 딕셔너리 생성: {범주: 정수 인코딩 값}
    mapping = {category: idx for idx, category in enumerate(le.classes_)}
    
    # 테스트 데이터에 대해 mapping을 적용
    # 만약 해당 값이 mapping에 없으면 -1을 할당 (원하는 기본값으로 변경 가능)
    test_df[col] = test_df[col].map(lambda x: mapping.get(x, -1))

#### 오버샘플링

In [9]:
oversampler = RandomOverSampler(random_state=42)
X_res, y_res = oversampler.fit_resample(X, y)

In [10]:
# 원본 데이터에서 타겟 값 비율(%) 추출
target_ratio = train_df["임신 성공 여부"].value_counts(normalize=True) * 100
print("원본 데이터 타겟 비율 (%):")
print(target_ratio)

# 오버샘플링 후 데이터에서 타겟 값 비율(%) 추출
target_ratio_res = y_res.value_counts(normalize=True) * 100
print("\n오버샘플링 후 타겟 비율 (%):")
print(target_ratio_res)


원본 데이터 타겟 비율 (%):
임신 성공 여부
0    74.16511
1    25.83489
Name: proportion, dtype: float64

오버샘플링 후 타겟 비율 (%):
임신 성공 여부
0    50.0
1    50.0
Name: proportion, dtype: float64


In [31]:
X_train, X_val, y_train, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

### Model

In [51]:
model = RandomForestClassifier(random_state=42)

In [41]:
xgb_model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

In [47]:
lgbm_model = LGBMClassifier(random_state=42)

#### Train

In [52]:
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

#### Predict

In [53]:
y_train_pred_proba = model.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_pred_proba > 0.5).astype(int)

In [54]:
# 학습 데이터 평가 
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))
print("AUC PR:", average_precision_score(y_train, y_train_pred_proba))
print("ROC AUC:", roc_auc_score(y_train, y_train_pred_proba))

Accuracy: 0.9782738760535971
              precision    recall  f1-score   support

           0       0.99      0.97      0.98    152013
           1       0.97      0.99      0.98    152183

    accuracy                           0.98    304196
   macro avg       0.98      0.98      0.98    304196
weighted avg       0.98      0.98      0.98    304196

AUC PR: 0.9977239027081548
ROC AUC: 0.9977841612940705


In [55]:
# 검증 세트에 대해 예측 확률(양성 클래스: index 1) 산출
y_val_pred_proba = model.predict_proba(X_val)[:, 1]
# 임계값 0.5를 적용하여 이진 예측 생성
y_val_pred = (y_val_pred_proba > 0.5).astype(int)

In [56]:
# 검증 데이터 평가
accuracy = accuracy_score(y_val, y_val_pred)
print("Accuracy:", accuracy)
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_pred_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_pred_proba))

Accuracy: 0.8430374753451677
              precision    recall  f1-score   support

           0       0.91      0.77      0.83     38110
           1       0.80      0.92      0.85     37940

    accuracy                           0.84     76050
   macro avg       0.85      0.84      0.84     76050
weighted avg       0.85      0.84      0.84     76050

AUC PR: 0.9221269329160322
ROC AUC: 0.9332203611967522


### 전체 데이터로 모델 재학습

In [25]:
model_full = RandomForestClassifier(random_state=42)
model_full.fit(X_res, y_res)

RandomForestClassifier(random_state=42)

In [26]:
# 테스트 세트에 대해 예측 확률 산출 (양성 클래스에 대한 확률)
y_test_pred_proba = model_full.predict_proba(test_df)[:, 1]

### Submission

In [27]:
sample_submission = pd.read_csv('Data/sample_submission.csv')
sample_submission['probability'] = y_test_pred_proba

In [28]:
sample_submission.to_csv('./submit.csv', index=False)